In [3]:
import sys                         # Manejo de llamadas del sistema
import os                          # Manejo de directorios
from datetime import datetime      # Manejo de tiempo
import SecuenciaARN                # Módulo SecuenciaARN con las clases: SecuenciaARN y SecuenciaAminoacido 
from importlib import reload       # Clase para recargar el módulo cuando se hace cambios

# En caso de que haya habido cambios en el módulo SecuenciaARN, aplica los cambios
reload(SecuenciaARN)

# ------------------------------- Directorios y ficheros

# Directorio de los ficheros input
st_dirEntrada = ".\\entrada"

# Directorio de los ficheros output
st_dirSalida = ".\\salida"

# Fichero FASTA a cargar
st_fichFasta = "ejemploFASTA.txt"

# Fichero de salida
st_fichSalida = "ejemploFASTA_traducido_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".txt"

# Nombre completo fichero FASTA
st_fasta= os.path.join (st_dirEntrada, st_fichFasta)

# Nombre completo fichero salida
st_salida= os.path.join (st_dirSalida, st_fichSalida)

# ------------------------------- Inicio del programa

# Mostramos información de la ejecución del proceso
print (f"***************************************************")
print (f"\tInicio del proceso")
print (f"\t\tDirectorio de ejecucion: " + os.getcwd())
print (f"\t\tFichero FASTA: " + st_fasta)
print (f"***************************************************\n")

print (f"\t1. Comprobaciones iniciales")

# Si el directorio de entrada no existe o el fichero FASTA no existe, mostramos un error
print (f"\t\t1.1. Directorio de entrada:")
if os.path.isdir(st_dirEntrada):
    print(f"\t\t\tDirectorio: {st_dirEntrada} OK")
else:
    print (f"\t\t\tERROR!!!! El directorio: {st_dirEntrada} no existe")
    sys.exit()

# Si el fichero FASTA existe, lo leemos y guardamos las secuencias ARN en los objetos secuenciasAminoacidos(secuenciasARN)
print (f"\t\t1.2. Fichero FASTA:")
if os.path.exists(st_fasta):
    print(f"\t\t\tFichero FASTA: {st_fasta}...OK. Cargando fichero...")

    st_fichFastaLeido = ""                # Almacenar el fichero FASTA leido
    lista_obj_secuenciaAminoacido = []    # Lista para almacenar los objetos SecuenciaAminoacido, con las secuencias ARN del fichero FASTA
    with open (st_fasta, "r") as open_fichFasta:
        # Guardamos el contenido completo del fichero FASTA.
        st_fichFastaLeido = open_fichFasta.read()

        # Al leer el fichero, nos quedamos al final del fichero. Para volver a leerlo, me vuelvo al principio
        open_fichFasta.seek(0)
        
        # Si es una secuencia ARN, la guardamos en una lista dentro de un objeto SecuenciaARN
        for linea in open_fichFasta:
            if linea.strip() != "" and not linea.startswith(">"):
                lista_obj_secuenciaAminoacido.append(SecuenciaARN.SecuenciaAminoacido (linea.strip()))

    print(f"\t\t\t\tFichero cargado\n")
else:
    print (f"\t\tERROR!!!! El fichero " + st_fasta + " no existe")
    sys.exit()

print (f"---------------------------------------------------\n")
print(f"\t2. Imprimimos por pantalla el fichero FASTA cargado")

print (f"\t\t2.1 Contenido:")
print (f"\n++++++++++++++++++++++++++ Contenido del fichero ++++++++++++++++++++++++++\n")
print (st_fichFastaLeido)
print (f"\n++++++++++++++++++++++++++ Fin contenido del fichero ++++++++++++++++++++++++++\n")

print (f"---------------------------------------------------\n")
print(f"\t3. Se filtra el contenido del fichero FASTA y nos quedamos solo con las secuencias ARN")
print (f"\t\t3.1 Numero total de secuencias ARN: {len(lista_obj_secuenciaAminoacido)}")
for i in range(0, len(lista_obj_secuenciaAminoacido), 1):
    print (f"\t\t\tSecuencia ARN {i+1}: {lista_obj_secuenciaAminoacido[i].st_secuenciaARN}")

print (f"\n---------------------------------------------------\n")
print (f"\t4. Traducción de la secuencia ARN")
print (f"\t\t4.1 Traducción en aminoacidos y lo mostramos por pantalla")

# Traducimos el ARN en aminoacidos. El traductor es un método de clase de la clase secuenciaAminoacido
for i in range (0, len(lista_obj_secuenciaAminoacido), 1):  
    print (f"\n\t\t\tTraduciendo la secuencia: {i+1} de ARN\n")
    lista_obj_secuenciaAminoacido[i].traduceARNtoAminoacido()
    lista_obj_secuenciaAminoacido[i].print_info()

print (f"---------------------------------------------------\n")
print (f"\t5. Escribimos la información en el fichero de salida\n")

# Escribimos en el fichero de salida, la secuencia ARN y la traducida a AA.
# Si el directorio de salida no existe, lo creamos
print (f"\t\t5.1. Directorio de salida:")

if os.path.isdir(st_dirSalida):
    print(f"\t\t\tDirectorio: {st_dirSalida} OK")
else:
    print(f"\t\tCreamos el directorio:{st_dirSalida}")
    os.mkdir(st_dirSalida)
    print(f"\t\t\tDirectorio:{st_dirSalida} creado")

print (f"\t\t5.2. Escribimos la relación ARN-AA en el fichero de salida: {st_salida}")

# Para escribir el fichero de salida en modo binario, usamos "wb" y es necesrio indicar la codificación a usar. Usamos "utf-8".
try:
    with open (st_salida, "wb") as open_salida:
        for i in range(0,len(lista_obj_secuenciaAminoacido), 1):
            st_texto_cod = lista_obj_secuenciaAminoacido[i].st_secuenciaARN + ";" + lista_obj_secuenciaAminoacido[i].st_secuenciaAminoacido + "\n"
            open_salida.write (st_texto_cod.encode("utf-8"))
except IOError:
    print(f"\t\t\tERROR!!! No se ha podido escribir correctamente el fichero de salida")
else:
    print(f"\t\t\tCompletado")

print (f"---------------------------------------------------\n")
print (f"\t6. Leemos el fichero generado y lo mostramos por pantalla\n")

print (f"\t\t6.1. Comprobamos que el fichero de salida existe")

if os.path.isfile(st_salida) and os.path.getsize(st_salida) > 0:
    print(f"\t\t\tEl fichero: {st_salida} existe y tiene contenido")
else:
    print(f"\t\t\tERROR!!! No se ha generado el fichero de salida")

print (f"\t\t6.2. Leemos y mostramos por pantalla")

with open (st_salida, "r") as open_salida:
    # Leemos el contenido
    print (f"\n++++++++++++++++++++++++++ Contenido del fichero ++++++++++++++++++++++++++\n")
    print (open_salida.read())
    print (f"\n++++++++++++++++++++++++++ Fin contenido del fichero ++++++++++++++++++++++++++\n")



print (f"\n***************************************************")
print (f"\tFin del proceso")
print (f"***************************************************")





***************************************************
	Inicio del proceso
		Directorio de ejecucion: C:\Users\psanz\Desktop\master\fundamentos de programacion\traductorConClases
		Fichero FASTA: .\entrada\ejemploFASTA.txt
***************************************************

	1. Comprobaciones iniciales
		1.1. Directorio de entrada:
			Directorio: .\entrada OK
		1.2. Fichero FASTA:
			Fichero FASTA: .\entrada\ejemploFASTA.txt...OK. Cargando fichero...
				Fichero cargado

---------------------------------------------------

	2. Imprimimos por pantalla el fichero FASTA cargado
		2.1 Contenido:

++++++++++++++++++++++++++ Contenido del fichero ++++++++++++++++++++++++++

>NM_004004.6 Homo sapiens gap junction protein beta 2 (GJB2), mRNA
GUUGCGGCCCCGCAGCGCCCGCGCGCUCCUCUCCCCGACUCGGAGCCCCUCGGCGGCGCCAA

>XM_011535049.3 Homo sapiens gap junction protein beta 2 (GJB2), transcript variant X1, mRNA
GUUGCGGCCCCGCAGCGCCCGCGCGCUCCACUCGGAGCCCCUCGGCGGCGCCCGGCCCAGGA

>NM_004004.6 MUT
GUUGCGGCCCCGCAGCACCC